In [7]:
import pandas as pd
import numpy as np

# Make DataFrame outputs easier to read in the notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [8]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/content/beijing_data/PRSA_Data_20130301-20170228")

# Find Aotizhongxin station file
file_path = next(DATA_DIR.glob("*Aotizhongxin*.csv"))

print("Loading file from:", file_path)

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

Loading file from: /content/beijing_data/PRSA_Data_20130301-20170228/PRSA_Data_Aotizhongxin_20130301-20170228.csv
Dataset shape: (35064, 18)
Columns: ['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


In [9]:
print("Dataset shape:", df.shape)
df.columns

Dataset shape: (35064, 18)


Index(['No', 'year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN',
       'wd', 'WSPM', 'station'],
      dtype='object')

In [10]:
# Drop columns that do not add predictive value
df = df.drop(columns=["No", "station"])

In [11]:
# Combine year, month, day, and hour into a single datetime column
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

# Remove the original time columns
df = df.drop(columns=["year", "month", "day", "hour"])

# Set datetime as index and ensure chronological order
df = df.set_index("datetime")
df = df.sort_index()

df.head()

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM
datetime,,,,,,,,,,,,
2013-03-01 00:00:00,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4
2013-03-01 01:00:00,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7
2013-03-01 02:00:00,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6
2013-03-01 03:00:00,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1
2013-03-01 04:00:00,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0


In [12]:
print("Start date:", df.index.min())
print("End date:", df.index.max())
print("Total hourly records:", len(df))

Start date: 2013-03-01 00:00:00
End date: 2017-02-28 23:00:00
Total hourly records: 35064


In [13]:
# Count missing values per column
df.isna().sum().sort_values(ascending=False)

,0
CO,1776
O3,1719
NO2,1023
SO2,935
PM2.5,925
PM10,718
wd,81
TEMP,20
PRES,20
DEWP,20


In [14]:
# Calculate percentage of missing values per column
na_count = df.isna().sum()
na_percentage = (na_count / len(df)) * 100

na_summary = (
    pd.DataFrame({
        "Missing Count": na_count,
        "Missing Percentage (%)": na_percentage.round(2)
    })
    .sort_values("Missing Percentage (%)", ascending=False)
)

na_summary

,Missing Count,Missing Percentage (%)
CO,1776,5.07
O3,1719,4.90
NO2,1023,2.92
SO2,935,2.67
PM2.5,925,2.64
PM10,718,2.05
wd,81,0.23
TEMP,20,0.06
PRES,20,0.06
DEWP,20,0.06


In [15]:
pollutant_features = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
weather_features = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM", "wd"]

print("Pollutant features:", pollutant_features)
print("Weather features:", weather_features)

Pollutant features: ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
Weather features: ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'wd']


In [17]:
from pathlib import Path

OUTPUT_PATH = Path("/content/beijing_data/processed/aotizhongxin_baseline.csv")

# Create folder if it does not exist
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save dataset
df.to_csv(OUTPUT_PATH, index=False)

print(f"Baseline dataset saved to: {OUTPUT_PATH}")

Baseline dataset saved to: /content/beijing_data/processed/aotizhongxin_baseline.csv
